# Semana 3 · ¿Cómo digo dónde está cada cosa en el cielo?

Este cuaderno acompaña a la sesión de proyecto de la semana 3: un mapa de estrellas brillantes en coordenadas ecuatoriales y el cálculo de cuándo y a qué altura se ve cada una desde tu latitud.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. Los datos: estrellas brillantes

La **ascensión recta** (AR) se mide en horas, minutos y segundos (24 h = 360°) y la **declinación** (Dec) en grados y minutos de arco. Los valores son de referencia, para la época J2000 y redondeados al minuto: **no los he podido contrastar con un catálogo desde mi entorno**. Comprobar unos pocos en Stellarium es parte de la sesión de astronomía, y si alguno difiere más de uno o dos minutos, fíate de Stellarium y corrige la tabla.

In [ ]:
# (nombre, AR horas, AR minutos, Dec grados, Dec minutos de arco)  -- Dec con signo en los grados
estrellas = [
    ("Sirio",       6, 45, -16, 43),
    ("Arturo",     14, 16,  19, 11),
    ("Vega",       18, 37,  38, 47),
    ("Capella",     5, 17,  46,  0),
    ("Rigel",       5, 15,  -8, 12),
    ("Procyon",     7, 39,   5, 14),
    ("Betelgeuse",  5, 55,   7, 24),
    ("Altair",     19, 51,   8, 52),
    ("Aldebarán",   4, 36,  16, 31),
    ("Antares",    16, 29, -26, 26),
    ("Spica",      13, 25, -11, 10),
    ("Deneb",      20, 41,  45, 17),
    ("Régulo",     10,  8,  11, 58),
    ("Pólux",       7, 45,  28,  2),
    ("Polar",       2, 32,  89, 16),
]

def ar_a_grados(h, m):
    """Ascensión recta en horas y minutos a grados (1 h = 15°)."""
    return (h + m / 60) * 15

def dec_a_grados(g, m):
    """Declinación en grados y minutos de arco a grados decimales (respeta el signo)."""
    signo = -1 if g < 0 else 1
    return signo * (abs(g) + m / 60)

datos = [(n, ar_a_grados(h, m), dec_a_grados(g, mm)) for n, h, m, g, mm in estrellas]
for n, ar, dec in datos[:4]:
    print(f"{n:12s} AR = {ar:7.2f}°   Dec = {dec:+7.2f}°")

Ojo con la declinación de Rigel: está en −8° 12′, o sea, −8,2°, no −7,8°. Por eso la función conserva el signo de los grados y suma los minutos en valor absoluto.

## 2. El mapa del cielo en el plano

Cada estrella es un punto en un plano: la AR en el eje horizontal y la Dec en el vertical. Por convención, en los mapas del cielo la AR crece **hacia la izquierda** (el este queda a la izquierda cuando miras al sur), así que invertimos el eje horizontal.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for n, ar, dec in datos:
    ax.plot(ar / 15, dec, "o", color="#d9a21e")
    ax.annotate(n, (ar / 15, dec), textcoords="offset points", xytext=(5, 4), fontsize=9)
ax.set_xlim(24, 0)                      # la AR crece hacia la izquierda
ax.set_ylim(-40, 95)
ax.set_xticks(range(0, 25, 2))
ax.set_xlabel("ascensión recta (horas)")
ax.set_ylabel("declinación (°)")
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_title("Estrellas brillantes en coordenadas ecuatoriales (J2000)")
ax.grid(alpha=0.3)
plt.show()

## 3. Cuándo y a qué altura se ve cada una

Una estrella cruza el meridiano (el arco que va del norte al sur pasando por el cenit) una vez por día, y en ese momento está lo más alta posible. Para un observador en el hemisferio norte de latitud $\varphi$, esa **altura máxima** es

$$h_{\max} = 90^\circ - |\varphi - \delta|$$

donde $\delta$ es la declinación de la estrella. Dos consecuencias: una estrella con $\delta > 90^\circ - \varphi$ nunca se pone (es **circumpolar**) y una con $\delta < -(90^\circ - \varphi)$ nunca sale.

In [ ]:
mi_latitud = 40   # grados, positiva en el hemisferio norte. Cámbiala por la tuya.

def altura_maxima(dec, lat):
    return 90 - abs(lat - dec)

def clasificar(dec, lat):
    if dec > 90 - lat:
        return "circumpolar"
    if dec < -(90 - lat):
        return "nunca sale"
    return "sale y se pone"

print(f"Latitud {mi_latitud}°\n")
for n, ar, dec in sorted(datos, key=lambda t: -t[2]):
    print(f"{n:12s} Dec {dec:+6.1f}°   altura máxima {altura_maxima(dec, mi_latitud):5.1f}°   {clasificar(dec, mi_latitud)}")

La Polar tiene $\delta \approx 89{,}3^\circ$, así que su altura es $90 - |\varphi - 89{,}3| = \varphi + 0{,}7^\circ$ aproximadamente: **la altura de la Polar es tu latitud** (con un error de menos de 1°). Es la forma más antigua de medir la latitud.

## 4. Compara con Stellarium

En Stellarium (con tu latitud), anota la altura máxima de tres estrellas de la tabla y escríbela aquí.

In [ ]:
medidas_stellarium = {"Vega": None, "Sirio": None, "Capella": None}   # alturas máximas en grados

for n, ar, dec in datos:
    if n in medidas_stellarium:
        modelo = altura_maxima(dec, mi_latitud)
        medida = medidas_stellarium[n]
        if medida is None:
            print(f"{n:10s} modelo {modelo:5.1f}°  (sin medida)")
        else:
            print(f"{n:10s} modelo {modelo:5.1f}°  Stellarium {medida:5.1f}°  diferencia {medida - modelo:+.1f}°")

## 5. Para pensar

1. Cambia `mi_latitud` a 0 y a 60. ¿Qué estrellas son circumpolares en cada caso? ¿Y a 90°?
2. ¿Desde qué latitud se ve Antares a más altura? ¿Por qué desde Europa se ve siempre baja?
3. Dos estrellas con la misma AR y declinaciones 20° y 60°, ¿cruzan el meridiano a la vez? ¿A la misma altura?